In [35]:
import pandas as pd
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.metrics import roc_auc_score

In [36]:
df = pd.read_csv("../data/processed_telco_churn.csv")

X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y, 
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [37]:
rf = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_rf.fit(X_train, y_train) 

best_rf = grid_rf.best_estimator_

pred_rf_tuned = best_rf.predict(X_test)

rf_tuned_acc = accuracy_score(y_test, pred_rf_tuned)

rf_tuned_results = pd.DataFrame({
    "Model": ["Random Forest (Tuned)"],
    "Accuracy": [rf_tuned_acc],
    "Precision": [precision_score(y_test, pred_rf_tuned)],
    "Recall": [recall_score(y_test, pred_rf_tuned)],
    "F1 Score": [f1_score(y_test, pred_rf_tuned)],
    "Best Parameters": [grid_rf.best_params_]
})

rf_tuned_results

,Model,Accuracy,Precision,Recall,F1 Score,Best Parameters
0,Random Forest (Tuned),0.796731,0.64966,0.510695,0.571856,"{'max_depth': None, 'min_samples_leaf': 2, 'mi..."


In [38]:
gb = GradientBoostingClassifier(random_state=42)

param_grid = {
    "n_estimators": [100, 200, 500],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7]
}

grid_gb = GridSearchCV(
    estimator=gb,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid_gb.fit(X_train, y_train)

best_gb = grid_gb.best_estimator_

pred_gb = best_gb.predict(X_test)

gb_tuned_results = pd.DataFrame({
    "Model": ["Gradient Boosting (Tuned)"],
    "Accuracy": [accuracy_score(y_test, pred_gb)],
    "Precision": [precision_score(y_test, pred_gb)],
    "Recall": [recall_score(y_test, pred_gb)],
    "F1 Score": [f1_score(y_test, pred_gb)],
    "Best Parameters": [grid_gb.best_params_]
})

gb_tuned_results

,Model,Accuracy,Precision,Recall,F1 Score,Best Parameters
0,Gradient Boosting (Tuned),0.793888,0.632911,0.534759,0.57971,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti..."


In [39]:
xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

grid_xgb = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid_xgb.fit(X_train, y_train)

print(grid_xgb.best_params_)

best_xgb = grid_xgb.best_estimator_

pred_xgb = best_xgb.predict(X_test)

xgb_tuned_results = pd.DataFrame({
    "Model": ["XGBoost (Tuned)"],
    "Accuracy": [accuracy_score(y_test, pred_xgb)],
    "Precision": [precision_score(y_test, pred_xgb)],
    "Recall": [recall_score(y_test, pred_xgb)],
    "F1 Score": [f1_score(y_test, pred_xgb)],
    "Best Parameters": [grid_xgb.best_params_]
})

xgb_tuned_results


{'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}


,Model,Accuracy,Precision,Recall,F1 Score,Best Parameters
0,XGBoost (Tuned),0.790334,0.624606,0.529412,0.573082,"{'colsample_bytree': 0.8, 'learning_rate': 0.1..."


In [ ]:
final_results = pd.concat([
    rf_tuned_results,
    gb_tuned_results,
    xgb_tuned_results
], ignore_index=True)

final_results.sort_values(
    by="F1 Score",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score,Best Parameters
1,Gradient Boosting (Tuned),0.793888,0.632911,0.534759,0.579710,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti..."
2,XGBoost (Tuned),0.790334,0.624606,0.529412,0.573082,"{'colsample_bytree': 0.8, 'learning_rate': 0.1..."
0,Random Forest (Tuned),0.796731,0.649660,0.510695,0.571856,"{'max_depth': None, 'min_samples_leaf': 2, 'mi..."
